<div style="border:1px solid #d9e1ea;border-left:6px solid #1f6fad;border-radius:14px;padding:18px 20px;background:#fff;"><h1 style="margin:0 0 6px;color:#16213b;">FastAPI — Complete Learning Notebook</h1><p style="margin:0;color:#63738a;">ASGI, routes, validation, sync/async, dependency injection, errors, testing and Docker — mapped to app/main.py.</p><p style="margin:10px 0 0;"><code>notebooks/fastapi/fastapi_project_learning.ipynb</code></p></div>

![Request](assets/01_request.svg)

## 1. What FastAPI ？

FastAPI is a modern Python framework for building web APIs.

It lets other applications call your Python logic through HTTP.

In your project:
```
Client / Frontend / Bot
        ↓
     FastAPI
        ↓
ReservationAgent
        ↓
LangGraph / RAG / Analytics
        ↓
     Response
```

## 2. FastAPI vs Uvicorn

**Uvicorn** is a lightweight **ASGI** (Asynchronous Server Gateway Interface) server used to run Python web applications such as FastAPI.

**FastAPI** is the web framework; **Uvicorn is the server** that runs it.

```
Browser / Client
      ↓
   Uvicorn
      ↓
   FastAPI
      ↓
Your Python Logic
```

![ASGI](assets/02_asgi.svg)

`uvicorn app.main:app --reload` means: start Uvicorn, import `app.main`, then run the `app = FastAPI(...)` object.

## 3. Current app/main.py

In [2]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="Reservation Analytics AI Agent")

class AskRequest(BaseModel):
    question: str

@app.get("/health")
def health() -> dict:
    return {"status": "ok", "backend": settings.backend}

@app.post("/ask")
def ask(request: AskRequest) -> dict:
    result = agent.invoke(request.question)
    return {
        "answer": result.get("answer", ""),
        "route": result.get("route", ""),
        "status": result.get("status", ""),
    }

## 4. HTTP Methods

GET reads status/resources; POST submits data/processing; PUT/PATCH update; DELETE removes.

## 5. Pydantic Request Validation

In [ ]:
class AskRequest(BaseModel):
    question: str

## 6. Sync vs Async

Use `async def` when the underlying I/O is truly asynchronous. Blocking clients do not become non-blocking just because the endpoint is async.

## 7. Dependency Injection

In [ ]:
from fastapi import Depends

def get_agent():
    return agent

@app.post("/ask")
def ask(request: AskRequest, agent=Depends(get_agent)):
    return agent.invoke(request.question)

## 8. Response Models and Errors

In [ ]:
from fastapi import HTTPException

try:
    result = agent.invoke(request.question)
except RuntimeError as exc:
    raise HTTPException(status_code=500, detail=str(exc))

## 9. Middleware / CORS / Lifespan

Middleware wraps requests; CORS controls browser cross-origin access; lifespan handles startup/shutdown resources.

## 10. Testing

In [ ]:
from fastapi.testclient import TestClient
from app.main import app

client = TestClient(app)
assert client.get("/health").status_code == 200

## 11. Docker Serving

In [ ]:
uvicorn app.main:app --host 0.0.0.0 --port 8000

## Q&A — Fast Review

<details open><summary><b>Q1. FastAPI vs Uvicorn?</b></summary>

**Answer:** FastAPI is the web framework; Uvicorn is the ASGI server.

</details>

<details open><summary><b>Q2. Why Pydantic?</b></summary>

**Answer:** It validates typed request/response contracts.

</details>

<details open><summary><b>Q3. Should every endpoint be async?</b></summary>

**Answer:** No, only when the underlying I/O is asynchronous.

</details>

<details open><summary><b>Q4. Where is LangGraph?</b></summary>

**Answer:** FastAPI receives the HTTP request, then ReservationAgent/LangGraph controls the internal workflow.

</details>

<details open><summary><b>Q5. Production improvements?</b></summary>

**Answer:** Response models, auth, DI, logging, tracing, timeouts, lifespan initialization.

</details>

## Classic Architecture Q&A — Memorize This

### Q. Why do you use LangChain selectively instead of making it the entire application framework?

> **I use LangChain selectively rather than making it the entire application framework. LangChain's ChatOpenAI integration handles structured extraction, LangGraph handles stateful workflow orchestration, and LlamaIndex with FAISS handles the knowledge RAG layer. This keeps responsibilities explicit and prevents the LLM from directly controlling analytics SQL.**

<div style="background:#eef7ff;border:1px solid #c9e0f2;border-radius:10px;padding:10px 12px;margin:10px 0;">
</div>

### Memory Map

```text
LangChain / ChatOpenAI  → Structured Extraction
Pydantic                → Typed Contract
LangGraph               → Stateful Workflow
LlamaIndex + FAISS      → Knowledge RAG
Controlled SQL          → Trusted Numbers
FastAPI                 → Service API
```

### One-line takeaway

> **Do not force every responsibility into one framework. Keep the boundaries explicit.**